In [ ]:
from matplotlib import pyplot as plt
from scipy.stats import norm
import numpy as np 
import csv
import math
import sys
import VSPFunctions as vsp
import pandas as pd
import seaborn as sns
from scipy.optimize import curve_fit
from scipy.stats import lognorm
import os

directory = "/users/cdcook/VSP/graphics/"

sns.set_theme(style="darkgrid")

# Function to check if a specific flag is true
def flag_is_true(row, flag):
    return row.get(flag, False)

field = 'sky0001_1d'
night = '0824'

In [ ]:
exposure = 0
title = f'/lustre/work/client/users/cdcook/VSPData/{field}00{night}_exp' + str(exposure) + '.csv'
#title = f'/lustre/work/client/users/cdcook/VSPData/meanFields/mean{field}{night}Files/{field}00{night}_exp' + str(exposure) + '.csv' #sky0001_1c000928_exp0.csv
panDF = pd.read_csv(title)
panDF = panDF.drop_duplicates(subset=['RA', 'Dec'])
panDF = panDF[(panDF['Mag'] >= 5) & (panDF['Mag'] <= 25)]
panDF = panDF[panDF['gMeanPSFMag'] != -999]
panDF = panDF[panDF['rMeanPSFMag'] != -999]
panDF = panDF[panDF['iMeanPSFMag'] != -999]
panDF = panDF[panDF['zMeanPSFMag'] != -999]
panDF = panDF[panDF['yMeanPSFMag'] != -999]
panDF = panDF[panDF['gMeanKronMag'] != -999]
panDF = panDF[panDF['rMeanKronMag'] != -999]
panDF = panDF[panDF['iMeanKronMag'] != -999]
panDF = panDF[panDF['zMeanKronMag'] != -999]
panDF = panDF[panDF['yMeanKronMag'] != -999]

# Define a function to calculate the flux
def calculate_flux(magnitude):
    return np.power(10, (magnitude + 48.6) / -2.5)

# Calculate flux for each band
panDF['gflux'] = panDF['gMeanPSFMag'].apply(calculate_flux)
panDF['rflux'] = panDF['rMeanPSFMag'].apply(calculate_flux)
panDF['iflux'] = panDF['iMeanPSFMag'].apply(calculate_flux)
panDF['zflux'] = panDF['zMeanPSFMag'].apply(calculate_flux)
panDF['yflux'] = panDF['yMeanPSFMag'].apply(calculate_flux)

# Calculate total flux
panDF['totalFlux'] = (
    panDF['gflux'] * 0.1212 + 
    panDF['rflux'] * 0.1463 + 
    panDF['iflux'] * 0.1435 + 
    panDF['zflux'] * 0.098 + 
    panDF['yflux'] * 0.0393
) / 0.5483

# Calculate logpart
panDF['logpart'] = np.log10(panDF['totalFlux'] / 3631e-23)

# Calculate pseudoBoloMag
panDF['pseudoBoloMag'] = -2.5 * panDF['logpart']

flux_columns = ['gflux', 'rflux', 'iflux', 'zflux', 'yflux']
panDF.drop(columns=flux_columns, inplace=True)

panDF['Difference'] = panDF['pseudoBoloMag'] - panDF['Mag']

panDF['objInfoFlag'] = panDF['objInfoFlag'].apply(lambda x: f'0x{x:08X}')
panDF['qualityFlag'] = panDF['qualityFlag'].apply(lambda x: f'0x{x:08X}')

print(panDF)

In [ ]:
# Define the objInfoFlag dictionary with swapped keys and values
objInfoFlags = {
    0x00000000: "DEFAULT",
    0x00000001: "FEW",
    0x00000002: "POOR",
    0x00000004: "ICRF_QSO",
    0x00000008: "HERN_QSO_P60",
    0x00000010: "HERN_QSO_P05",
    0x00000020: "HERN_RRL_P60",
    0x00000040: "HERN_RRL_P05",
    0x00000080: "HERN_VARIABLE",
    0x00000100: "TRANSIENT",
    0x00000200: "HAS_SOLSYS_DET",
    0x00000400: "MOST_SOLSYS_DET",
    0x00000800: "LARGE_PM",
    0x00001000: "RAW_AVE",
    0x00002000: "FIT_AVE",
    0x00004000: "FIT_PM",
    0x00008000: "FIT_PAR",
    0x00010000: "USE_AVE",
    0x00020000: "USE_PM",
    0x00040000: "USE_PAR",
    0x00080000: "NO_MEAN_ASTROM",
    0x00100000: "STACK_FOR_MEAN",
    0x00200000: "MEAN_FOR_STACK",
    0x00400000: "BAD_PM",
    0x00800000: "EXT",
    0x01000000: "EXT_ALT",
    0x02000000: "GOOD",
    0x04000000: "GOOD_ALT",
    0x08000000: "GOOD_STACK",
    0x10000000: "BEST_STACK",
    0x20000000: "SUSPECT_STACK",
    0x40000000: "BAD_STACK"
}

def read_flags(flag_value, flag_dict):
    """Function to read and interpret flag values based on the provided flag dictionary."""
    active_flags = {flag_dict[flag_hex]: (flag_value & flag_hex) != 0 for flag_hex in flag_dict}
    return {flag_name: active for flag_name, active in active_flags.items() if active}

def add_flag_descriptions(df, flag_col_name, flag_dict):
    """Add a description column to the DataFrame based on the flags."""
    descriptions = df[flag_col_name].apply(lambda x: read_flags(int(x, 16) if isinstance(x, str) else x, flag_dict))
    description_col_name = f"{flag_col_name}_Description"
    df[description_col_name] = descriptions
    return df

panDF = add_flag_descriptions(panDF, "objInfoFlag", objInfoFlags)

In [ ]:
def KronCut(kronBand, kronDist, dataDF):
    if (kronBand == 'g'):
        kronName = 'gMeanKronMag'
        psfName = 'gMeanPSFMag'
        kronCutName = 'gKron'
    if (kronBand == 'r'):
        kronName = 'rMeanKronMag'
        psfName = 'rMeanPSFMag'
        kronCutName = 'rKron'
    if (kronBand == 'i'):
        kronName = 'iMeanKronMag'
        psfName = 'iMeanPSFMag'
        kronCutName = 'iKron'
    if (kronBand == 'z'):
        kronName = 'zMeanKronMag'
        psfName = 'zMeanPSFMag'
        kronCutName = 'zKron'
    if (kronBand == 'y'):
        kronName = 'yMeanKronMag'
        psfName = 'yMeanPSFMag'
        kronCutName = 'yKron'
        
    # Condition 1: psfName - kronName should be less than kronDist
    condition1 = abs(dataDF[psfName] - dataDF[kronName]) < kronDist
    # Apply both conditions to the DataFrame
    dataDF = dataDF[condition1]
    
    return dataDF, kronName, psfName, kronCutName

def BitFlagCut(bitFlags, dataDF):
    dataDF = dataDF[dataDF['Flags'] == bitFlags]
    return dataDF

def ColorCut(dataDF, sub1, sub2):
    if (sub1 == 'gr'):
        band1a = 'gMeanPSFMag'
        band1b = 'rMeanPSFMag'
        band1name = "g-r"
        
    elif (sub1 == 'gi'):
        band1a = 'gMeanPSFMag'
        band1b = 'iMeanPSFMag'
        band1name = "g-i"
        
    elif (sub1 == 'ri'):
        band1a = 'rMeanPSFMag'
        band1b = 'iMeanPSFMag'
        band1name = "r-i"
        
    if (sub2 == 'gr'):
        band2a = 'gMeanPSFMag'
        band2b = 'rMeanPSFMag'
        band2name = "g-r"
    
    elif (sub2 == 'gi'):
        band2a = 'gMeanPSFMag'
        band2b = 'iMeanPSFMag'
        band2name = "g-i"
        
    elif (sub2 == 'ri'):
        band2a = 'rMeanPSFMag'
        band2b = 'iMeanPSFMag'
        band2name = "r-i"
        
    x = []
    y = []
    for index, row in dataDF.iterrows():
        x.append(row[band1a] - row[band1b])
        y.append(row[band2a] - row[band2b])

    dataDF.insert(0, 'color1', x)
    dataDF.insert(1, 'color2', y)
    return dataDF, band1a, band1b, band2a, band2b, band1name, band2name

In [ ]:
bitFlags = 0.0
color2 = 'gr'
color1 = 'ri'
kronBand = 'r'
kronDist = '0.5'

kronCutTF = False
bitFlagTF = False
colorCutTF = False


panDF, band1a, band1b, band2a, band2b, band1name, band2name = ColorCut(panDF, 'gr', 'ri')
print(len(panDF))

In [ ]:
panDF, kronName, psfName, kronCutName = KronCut(kronBand='g', kronDist = 0.5, dataDF = panDF)
kronCutTF = True
print(len(panDF))

In [ ]:
panDF2 = panDF
#panDF2 = panDF[panDF["ginfoFlag2_Description"].apply(lambda x: flag_is_true(x, "SATSTAR_PROFILE"))]
print(len(panDF2))
params = np.polyfit(panDF2['Mag'], panDF2['pseudoBoloMag'], 1, full=False, cov=True)   
x = np.linspace(8, 20, 10000)   
print("slope: ", params[0][0])
print("AB offset: ", params[0][1])
print("Total Count: ", panDF.shape[0])
plt.figure(figsize=(18,9))
ax = sns.scatterplot(x='Mag', y='pseudoBoloMag', data=panDF2)
plt.plot(x, vsp.oneDFit(params[0][0], params[0][1], x), 'r--')
#plt.plot(x, oneDFit(params[0][0], params[0][1], x) + 0.8, 'r--')
plt.xlim(8,20)
plt.ylim(8,22)
plt.xlabel('ROTSE Mag', fontsize=20)
plt.ylabel('Pan Mag', fontsize=20)
ax.text(9,18,"slope: %4.4f" % params[0][0], fontsize=20)
ax.text(9,17,"AB offset: %4.4f" % params[0][1], fontsize=20)
ax.text(9,16,"Total Count: %s" % panDF2.shape[0], fontsize=20)

text = "Mean Xtetrans_1b_0410 Psuedo-Bolometric zero point Exposure: " + str(exposure + 1)
fileName = "PB_zero_point"
if(kronCutTF):
    text = text + " || " + kronCutName + " +-" + kronDist
    fileName = fileName + "_" + kronCutName + kronDist
if(bitFlagTF):
    text = text + " || " + "E-flag cut"
    fileName = fileName + "_flagCut"
if(colorCutTF):
    text = text + " || " + "Color Cut"
    fileName = fileName + "_colorCut"

fileName = fileName + "NewPass4.png"

plt.title(text, fontsize = 20)
plt.show()

In [ ]:
plt.figure(figsize=(18, 9))
sns.scatterplot(x="color2", y="color1", data=panDF)
plt.xlim(-1,3)
plt.ylim(-1,3)

plt.xlabel(band2name, fontsize=20)
plt.ylabel(band1name, fontsize=20)

text = "Mean Xtetrans_1b_0410 Color Locus Plot Exposure: " + str(exposure + 1)
fileName = "CL"
if(kronCutTF):
    text = text + " || " + kronCutName + " +-" + kronDist
    fileName = fileName + "_" + kronCutName + kronDist

if(bitFlagTF):
    text = text + " || " + "E-flag cut"
    fileName = fileName + "_flagCut"

if(colorCutTF):
    text = text + " || " + "Color Cut"
    fileName = fileName + "_colorCut"

plt.title(text, fontsize = 20)
fileName = fileName + "NewPass4.png"

#plt.savefig(directory + fileName)

plt.show()

In [ ]:
def lognormFIT(x, sigma, loc , scale):
    return scale*lognorm.pdf(x, s=sigma, loc=loc)
    
def doubleGaussian(x, c1, mu1, sigma1, c2, mu2, sigma2):
    return c1*np.exp(-((x-mu1)**2)/(2*sigma1**2)) + c2*np.exp(-((x-mu2)**2)/(2*sigma2**2))

def Gaussian(x, c, mu, sigma):
    return c*np.exp(-((x-mu)**2)/(2*sigma**2))

plt.figure(figsize=(18,9))
ax = sns.histplot(x=panDF['Difference'], color="b", bins=500, stat='probability')
xlist = []
ylist = []
for i in ax.patches:
    if (i.get_x() > -1.0 and i.get_x() < 0.6):
        xlist.append(i.get_x())
        ylist.append(i.get_height())

popt, pcov = curve_fit(lognormFIT, xlist, ylist, p0=[0.15, -1, 100], maxfev=100000, bounds=([-5, -5, -100], [1, 1, 100]))
print(popt)

popt2, pcov2 = curve_fit(doubleGaussian, xlist, ylist, p0=[1, -1, 0.5, 1, -1, 0.5], maxfev=100000, bounds=([-100, -2, 0, -100, -2, 0], [1, 1, 100, 1, 1, 100]))

popt3, pcov3 = curve_fit(Gaussian, xlist, ylist, p0=[1, -1, 0.5], maxfev=100000, bounds=([-100, -2, 0], [100, 1, 1]))

yobv = ylist
yexpDG = doubleGaussian(xlist, popt2[0], popt2[1], popt2[2], popt2[3], popt2[4], popt2[5])
yexpLN = lognormFIT(xlist, popt[0], popt[1], popt[2])
yexpGA = Gaussian(xlist, popt3[0], popt3[1], popt3[2])
    
chisqLN = 0
for n in range(yobv.__len__()):
    if(xlist[n] > -0.5 and xlist[n] < 0.25):
        temp = ((yobv[n] - yexpLN[n])**2)/yexpLN[n]
        #print(temp)
        chisqLN = chisqLN + temp
    
chisqDG = 0
for n in range(yobv.__len__()):
    if(xlist[n] > -0.5 and xlist[n] < 0.25):
        temp = chisqDG + ((yobv[n] - yexpDG[n])**2)/yexpDG[n]
        #print(temp)
        chisqDG = chisqDG + temp

chisqGA = 0
for n in range(yobv.__len__()):
    if(xlist[n] > -0.5 and xlist[n] < 0.25):
        temp = chisqGA + ((yobv[n] - yexpGA[n])**2)/yexpGA[n]
        #print(temp)
        chisqGA = chisqGA + temp

print("Chi Square (Double Gaussian): ", chisqDG)
print("Chi Square (Lognormal): ", chisqLN)
print("Chi Sqaure (Gaussian): ", chisqGA)

p = lognorm.pdf(xlist, 1, 0, 1)

#plt.text(1,1,"Chi Square (Double Gaussian): %4.4f" % chisqDG)
#plt.text(1,1,"Chi Square (Lognormal): %4.4f" % chisqLN)
#plt.text(1,1,"Chi Square (Gaussian): %4.4f" % chisqGA)

plt.plot(xlist, lognormFIT(xlist, popt[0], popt[1], popt[2]), 'r--', label='Lognormal fit')
plt.plot(xlist, doubleGaussian(xlist, popt2[0], popt2[1], popt2[2], popt2[3], popt2[4], popt2[5]), 'k--', label='Double Gaussian Fit')
plt.plot(xlist, Gaussian(xlist, popt3[0], popt3[1], popt3[2]), 'p--', label='Gaussian Fit')
plt.xlim(-1,1)
plt.legend(labels=[
    "Lognormal fit: sigma = " + str.format('{0:.4f}', popt[0]) + ", loc = " + str.format('{0:.4f}', popt[1]) + ", scale = " + str.format('{0:.4f}', popt[2]),
    "Double Gaussian fit: c1 = " + str.format('{0:.4f}', popt2[0]) + ", mu1 = " + str.format('{0:.4f}', popt2[1]) + ", sigma1 = " + str.format('{0:.4f}', popt2[2]) + ", c2 = " + str.format('{0:.4f}', popt2[3]) + ", mu2 = " + str.format('{0:.4f}', popt2[4]) + ", sigma2 = " + str.format('{0:.4f}', popt2[5]),
    "Gaussian fit: c = " + str.format('{0:.4f}', popt3[0]) + ", mu = " + str.format('{0:.4f}', popt3[1]) + ", sigma = " + str.format('{0:.4f}', popt3[2])
])
plt.xlabel("diff(PanSTARRs, ROTSE)",  fontsize=20)
plt.ylabel("Probability",  fontsize=20)

text = "Mean Xtetrans_1b_0410 diff(PanSTARRs, Rotse) Histogram Exposure: " + str(exposure + 1)
fileName = "diffHist"
if(kronCutTF):
    text = text + " || " + kronCutName + " +-" + kronDist
    fileName = fileName + "_" + kronCutName + kronDist

if(bitFlagTF):
    text = text + " || " + "E-flag cut"
    fileName = fileName + "_flagCut"

if(colorCutTF):
    text = text + " || " + "Color Cut"
    fileName = fileName + "_colorCut"

plt.title(text, fontsize = 20)
fileName = fileName + "NewPass4.png"

#plt.savefig(directory + fileName)

plt.show()

__SUB IMAGING__

In [ ]:
def subimage_by_ra_dec_balanced(df, n_ra_div=3, n_dec_div=3):
    """Split the dataframe into subimages by evenly dividing the RA and Dec ranges."""
    ra_min, ra_max = df['RA'].min(), df['RA'].max()
    dec_min, dec_max = df['Dec'].min(), df['Dec'].max()
    
    ra_bins = np.linspace(ra_min, ra_max, n_ra_div + 1)
    dec_bins = np.linspace(dec_min, dec_max, n_dec_div + 1)
    
    subimages = []
    
    for i in range(n_ra_div):
        for j in range(n_dec_div):
            ra_mask = (df['RA'] >= ra_bins[i]) & (df['RA'] < ra_bins[i + 1])
            dec_mask = (df['Dec'] >= dec_bins[j]) & (df['Dec'] < dec_bins[j + 1])
            sub_df = df[ra_mask & dec_mask]
            subimages.append(sub_df)
    
    return subimages

# Split the data into subimages
subimages = subimage_by_ra_dec_balanced(panDF, n_ra_div=3, n_dec_div=3)

# Table to store RA/Dec limits, slope, AB offset, and count for each subimage
summary_table = []

for idx, sub_df in enumerate(subimages):
    ra_min, ra_max = sub_df['RA'].min(), sub_df['RA'].max()
    dec_min, dec_max = sub_df['Dec'].min(), sub_df['Dec'].max()
    print(f"Subimage {idx + 1}: RA limits ({ra_min}, {ra_max}), Dec limits ({dec_min}, {dec_max})")

    # Subimage processing and plotting similar to the main image
    plt.figure(figsize=(18, 9))
    ax = sns.scatterplot(x='Mag', y='pseudoBoloMag', data=sub_df)
    params = np.polyfit(sub_df['Mag'], sub_df['pseudoBoloMag'], 1, full=False, cov=True)
    x = np.linspace(8, 20, 10000)
    plt.plot(x, vsp.oneDFit(params[0][0], params[0][1], x), 'r--')
    plt.xlim(8, 20)
    plt.ylim(8, 22)
    plt.xlabel('ROTSE Mag', fontsize=20)
    plt.ylabel('Pan Mag', fontsize=20)
    ax.text(9, 18, "slope: %4.4f" % params[0][0], fontsize=20)
    ax.text(9, 17, "AB offset: %4.4f" % params[0][1], fontsize=20)
    ax.text(9, 16, "Total Count: %s" % sub_df.shape[0], fontsize=20)

    # Save subimage details to summary table
    summary_table.append({
        'Subimage': idx + 1,
        'RA_min': ra_min,
        'RA_max': ra_max,
        'Dec_min': dec_min,
        'Dec_max': dec_max,
        'Slope': params[0][0],
        'AB_offset': params[0][1],
        'Count': sub_df.shape[0]
    })

    text = f"Subimage {idx + 1} Mean Xtetrans_1b_0410 Psuedo-Bolometric zero point Exposure: {exposure + 1}"
    fileName = f"Subimage_{idx + 1}_PB_zero_point"
    if(kronCutTF):
        text = text + " || " + kronCutName + " +-" + kronDist
        fileName = fileName + "_" + kronCutName + kronDist
    if(bitFlagTF):
        text = text + " || " + "E-flag cut"
        fileName = fileName + "_flagCut"
    if(colorCutTF):
        text = text + " || " + "Color Cut"
        fileName = fileName + "_colorCut"

    fileName = fileName + "NewPass4.png"

    plt.title(text, fontsize=20)
    plt.show()

    plt.figure(figsize=(18, 9))
    sns.scatterplot(x="color2", y="color1", data=sub_df)
    plt.xlim(-1, 3)
    plt.ylim(-1, 3)

    plt.xlabel(band2name, fontsize=20)
    plt.ylabel(band1name, fontsize=20)

    text = f"Subimage {idx + 1} Mean Xtetrans_1b_0410 Color Locus Plot Exposure: {exposure + 1}"
    fileName = f"Subimage_{idx + 1}_CL"
    if(kronCutTF):
        text = text + " || " + kronCutName + " +-" + kronDist
        fileName = fileName + "_" + kronCutName + kronDist

    if(bitFlagTF):
        text = text + " || " + "E-flag cut"
        fileName = fileName + "_flagCut"

    if(colorCutTF):
        text = text + " || " + "Color Cut"

    plt.title(text, fontsize=20)
    fileName = fileName + "NewPass4.png"

    plt.show()

    plt.figure(figsize=(18, 9))
    ax = sns.histplot(x=sub_df['Difference'], color="b", bins=500, stat='probability')
    xlist = []
    ylist = []
    for i in ax.patches:
        if (i.get_x() > -1.0 and i.get_x() < 0.6):
            xlist.append(i.get_x())
            ylist.append(i.get_height())

    popt, pcov = curve_fit(lognormFIT, xlist, ylist, p0=[0.15, -1, 100], maxfev=100000, bounds=([-5, -5, -100], [1, 1, 100]))
    popt2, pcov2 = curve_fit(doubleGaussian, xlist, ylist, p0=[1, -1, 0.5, 1, -1, 0.5], maxfev=100000, bounds=([-100, -2, 0, -100, -2, 0], [1, 1, 100, 1, 1, 100]))
    popt3, pcov3 = curve_fit(Gaussian, xlist, ylist, p0=[1, -1, 0.5], maxfev=100000, bounds=([-100, -2, 0], [100, 1, 1]))

    yobv = ylist
    yexpDG = doubleGaussian(xlist, popt2[0], popt2[1], popt2[2], popt2[3], popt2[4], popt2[5])
    yexpLN = lognormFIT(xlist, popt[0], popt[1], popt[2])
    yexpGA = Gaussian(xlist, popt3[0], popt3[1], popt3[2])

    chisqLN = np.sum((np.array(yobv) - np.array(yexpLN))**2 / np.array(yexpLN))
    chisqDG = np.sum((np.array(yobv) - np.array(yexpDG))**2 / np.array(yexpDG))
    chisqGA = np.sum((np.array(yobv) - np.array(yexpGA))**2 / np.array(yexpGA))

    print(f"Subimage {idx + 1} Chi Square (Double Gaussian): {chisqDG}")
    print(f"Subimage {idx + 1} Chi Square (Lognormal): {chisqLN}")
    print(f"Subimage {idx + 1} Chi Square (Gaussian): {chisqGA}")

    plt.plot(xlist, lognormFIT(xlist, popt[0], popt[1], popt[2]), 'r--', label='Lognormal fit')
    plt.plot(xlist, doubleGaussian(xlist, popt2[0], popt2[1], popt2[2], popt2[3], popt2[4], popt2[5]), 'k--', label='Double Gaussian Fit')
    plt.plot(xlist, Gaussian(xlist, popt3[0], popt3[1], popt3[2]), 'p--', label='Gaussian Fit')
    plt.xlim(-1, 1)
    plt.legend(labels=[
        f"Lognormal fit: sigma = {popt[0]:.4f}, loc = {popt[1]:.4f}, scale = {popt[2]:.4f}",
        f"Double Gaussian fit: c1 = {popt2[0]:.4f}, mu1 = {popt2[1]:.4f}, sigma1 = {popt2[2]:.4f}, c2 = {popt2[3]:.4f}, mu2 = {popt2[4]:.4f}, sigma2 = {popt2[5]:.4f}",
        f"Gaussian fit: a = {popt3[0]:.4f}, mu = {popt3[1]:.4f}, sigma = {popt3[2]:.4f}"
    ])
    plt.xlabel("Pan-STARRS - ROTSE Mag", fontsize=20)
    plt.ylabel("Probability", fontsize=20)
    text = f"Subimage {idx + 1} Mean Xtetrans_1b_0410 Pseudo-Bolometric Difference (Pan - ROTSE) Exposure: {exposure + 1}"
    fileName = f"Subimage_{idx + 1}_PB_Difference"
    if(kronCutTF):
        text = text + " || " + kronCutName + " +-" + kronDist
        fileName = fileName + "_" + kronCutName + kronDist
    if(bitFlagTF):
        text = text + " || " + "E-flag cut"
        fileName = fileName + "_flagCut"
    if(colorCutTF):
        text = text + " || " + "Color Cut"
        fileName = fileName + "_colorCut"
    plt.title(text, fontsize=20)
    plt.show()

# Create a summary DataFrame from the table
summary_df = pd.DataFrame(summary_table)
print("\nSummary Table:")
print(summary_df)

csvName = "subimage_summary_0413_exp" + str(exposure) + ".csv"
csvName = "0413_exp" + str(exposure) + "subimage.csv"

# Save summary table to a CSV file if needed
summary_df.to_csv(csvName, index=False)